# Week 3 RAG Evaluation

This submission-facing notebook verifies the governed LangChain RAG assets and summarizes the frozen base-versus-RAG experiment. It does not download models or call an external API. Full Llama-3.1-8B-Instruct inference, BGE-M3 indexing, Chroma persistence, retrieval traces, and local Mistral/RAGAS diagnostics were executed on an NVIDIA A40 and archived in immutable private run directories.

**Claim boundary:** this is a public-data component-proxy evaluation for Fari and Senpai, not deployed-product performance.

In [1]:
from pathlib import Path
import json
import pandas as pd
import W03_RAG_Pipeline as rag_pipeline

ROOT = Path.cwd()
if not (ROOT / 'W03_RAG_Pipeline.py').exists():
    ROOT = ROOT / 'phase_b_evaluation'
ROOT

WindowsPath('D:/newIntern/yian-ingen-ai-eval/phase_b_evaluation')

## Frozen pipeline contract

- Candidate: `meta-llama/Llama-3.1-8B-Instruct` revision `0e9e39f249a16976918f6564b8830bc894c89659`
- Embedding: `BAAI/bge-m3` revision `5617a9f61b028005a4858fdac845db406aefb181`
- Vector database: persistent Chroma
- Framework: LangChain 1.x
- Seed: `42`; greedy BF16 decoding
- Evaluation: retrieval recall/MRR plus provisional local RAGAS-style relevance, faithfulness, and context coverage
- API boundary: no OpenAI API or other external model API

In [2]:
kb, eval_set, config = rag_pipeline.load_assets(
    ROOT / 'W03_RAG_Official_Knowledge_Base_v0.3.0.yaml',
    ROOT / 'W03_RAG_Official_Eval_Set_v0.3.0.yaml',
    ROOT / 'W03_RAG_Official_Run_Config_v0.3.0.yaml',
)
contract = rag_pipeline.validate_assets(kb, eval_set, config)
pd.Series(contract, name='value').to_frame()

,value
status,ok
pipeline_version,0.4.0
data_origin,official_public_curated
dataset_role,official_public_rag_benchmark
knowledge_base_id,w03_ingen_official_public
knowledge_base_version,0.3.0
evaluation_set_id,w03_ingen_official_fari_senpai
evaluation_set_version,0.3.0
documents,4
sections,16


The candidate-visible inputs exclude reference answers, weighted scoring points, forbidden claims, and evidence fact identifiers. Metadata filters admit only the registered owner, domain, access scope, confidentiality class, current status, and platform.

In [3]:
retrieval = pd.DataFrame([
    {'measure': 'Document recall@k', 'value': 1.0},
    {'measure': 'Evidence-fact recall@k', 'value': 1.0},
    {'measure': 'Hit@k', 'value': 1.0},
    {'measure': 'MRR', 'value': 1.0},
    {'measure': 'Metadata leakage', 'value': 0.0},
    {'measure': 'Mean retrieval latency (ms)', 'value': 103.049},
])
retrieval

,measure,value
0,Document recall@k,1.000
1,Evidence-fact recall@k,1.000
2,Hit@k,1.000
3,MRR,1.000
4,Metadata leakage,0.000
5,Mean retrieval latency (ms),103.049


In [4]:
base_vs_rag = pd.DataFrame([
    {'metric': 'Answer relevance', 'base': 0.343704, 'rag': 0.573415, 'rag_valid': '12/12'},
    {'metric': 'Faithfulness', 'base': None, 'rag': 0.818182, 'rag_valid': '11/12'},
    {'metric': 'Context relevance', 'base': None, 'rag': 0.479167, 'rag_valid': '12/12'},
    {'metric': 'Context coverage/recall', 'base': None, 'rag': 0.708333, 'rag_valid': '12/12'},
    {'metric': 'Context precision', 'base': None, 'rag': 0.590278, 'rag_valid': '12/12'},
])
base_vs_rag['rag_minus_base'] = base_vs_rag['rag'] - base_vs_rag['base']
base_vs_rag

,metric,base,rag,rag_valid,rag_minus_base
0,Answer relevance,0.343704,0.573415,12/12,0.229711
1,Faithfulness,NaN,0.818182,11/12,NaN
2,Context relevance,NaN,0.479167,12/12,NaN
3,Context coverage/recall,NaN,0.708333,12/12,NaN
4,Context precision,NaN,0.590278,12/12,NaN


## Interpretation

RAG increased provisional answer relevance by `0.229711` on the paired official-public benchmark. Retrieval was perfect on this small, metadata-isolated corpus, so that result establishes pipeline functionality rather than broad retrieval robustness. The retained prompt-only v0.3.1 run improved provisional faithfulness to `0.930556` with 12/12 finite rows while keeping retrieval byte-for-byte fixed. Later completeness prompts exposed a trade-off: more exhaustive compound answers could reintroduce unsupported negative claims or reduce aggregate relevance.

Automatic scores remain diagnostic because the local Judge is uncalibrated; the separate AI qualitative calibration reviews answer content directly. The three-model Week 2 replay is likewise diagnostic because the inherited Prometheus Judge failed calibration. See `W03_RAG_Official_Benchmark_Report.md`, `W03_RAG_AI_Calibration_Report.md`, `W03_RAG_Failure_Taxonomy.md`, and `W03_Evaluation_Memo.md` for the full validity and deployment analysis.